# Computer Vision: offline companion notebook
Run in a clean kernel. See SETUP.md. Read corresponding lessons and vary one parameter after each example.


## C01 — image processing
Dependencies: numpy opencv-python-headless.


In [ ]:
import numpy as np
import cv2
img=np.zeros((64,64,3),dtype=np.uint8)
cv2.rectangle(img,(16,16),(48,48),(0,0,255),-1)
gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
mask=cv2.threshold(gray,30,255,cv2.THRESH_BINARY)[1]
edges=cv2.Canny(gray,30,100)
assert mask[32,32]==255 and mask[0,0]==0
print('mask pixels',np.count_nonzero(mask),'edge pixels',np.count_nonzero(edges))

## C02 — cnn shapes
Dependencies: torch.


In [ ]:
import torch
from torch import nn
model=nn.Sequential(nn.Conv2d(3,8,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(8,3))
x=torch.randn(4,3,32,32)
assert model(x).shape==(4,3)
print('batch logits',model(x).shape)
# Replace synthetic tensors with audited, group-split labeled images for training.

## C03 — iou dice
Dependencies: numpy.


In [ ]:
import numpy as np
def box_iou(a,b):
    if a[2]<a[0] or a[3]<a[1] or b[2]<b[0] or b[3]<b[1]: raise ValueError('invalid xyxy')
    intersection=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))
    union=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-intersection
    return intersection/union if union else 0.
def dice(a,b):
    a,b=np.asarray(a,bool),np.asarray(b,bool)
    return 2*(a&b).sum()/(a.sum()+b.sum()) if a.any() or b.any() else 1.
assert box_iou((0,0,2,2),(1,1,3,3))==1/7
assert box_iou((0,0,2,2),(0,0,2,2))==1
assert dice([0,0],[0,0])==1
print('metrics passed')

## C04 — video motion
Dependencies: numpy opencv-python-headless.


In [ ]:
import cv2
import numpy as np
previous=np.zeros((64,64),np.uint8); current=previous.copy()
cv2.circle(previous,(25,25),7,255,-1); cv2.circle(current,(29,25),7,255,-1)
flow=cv2.calcOpticalFlowFarneback(previous,current,None,.5,3,15,3,5,1.2,0)
print('median object motion',np.median(flow[previous>0],axis=0))
assert flow.shape==(64,64,2)